# Exam model

I Mira Vorne assure that I have done the exercise independently and I have carefully followed all the rules of the exam. I understand serious consequences in case of acting against the rules

Let's see what the material looks like first. With random images with labels is good way to check out few images.

In [ ]:
import pickle
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, callbacks
import cv2

tallennustiedosto_testi = open('mnist.hupsista', 'rb')
[X_train, y_train] = pickle.load(tallennustiedosto_testi)
tallennustiedosto_testi.close()

X_train = X_train.astype('float32') / 255.0

from sklearn.model_selection import train_test_split

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=42)

print("x_test shape:", X_train.shape)
print("y_test shape:", y_train.shape)

# get random images with their labels to see what the data looks like
random_index = np.random.randint(0, len(X_train))

plt.imshow(X_train[random_index])
plt.title(f'Label: {y_train[random_index]}')
plt.show()

: 

Now we'll check all the labels used.

In [ ]:
# List all unique labels
unique_labels = np.unique(y_train)

print("Unique labels in y_train:", unique_labels)

The training/validation data is certainly not good... It seems like we have a challenging task at hand! I'll start by trying to design a machine learning model that can handle the variations of the data. 

After studying some options, I decided to start with a CNN model and sparse categorical crossentropy loss function. The model architecture is inspired by the simple and traditional LeNet-5 architecture, with few enhanchements. 

I will try to find optimal kernel size, so I don't use too much computing power, but will get good enough results. 

In the output layer, I'll use softmax activation function. In the multi-class classification problem, where each input can belong to one of many possible classes, the softmax function is apparently generally a good choice. 

I'm going to use Adam optimizer, which has the default learning rate is 0.001. I'll stick to the default for now. 

I also tried the tanh activation function, but ended up using ReLU, since it worked better.

Resources used for planning the first model:
* https://www.tensorflow.org/guide/keras/sequential_model
* https://www.tensorflow.org/tutorials/images/cnn
* https://keras.io/guides/training_with_built_in_methods/
* https://poloclub.github.io/cnn-explainer/

In [ ]:
# Define the model architecture
model = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)), #  Take the maximum value in each 2x2 window.
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(), # Convert from 2D matrix into a 1D vector, because Dense layers expect input in this format.
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])
# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Fit the model
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val), batch_size=32)

Not bad for the first try. Figures show a bit of overfitting. Let's plot some images for visual analysis.

I'm expecting to plot several times, so maybe it is good to make a plotting function now, so I can use it through the whole experiment.

In [ ]:
# Plot the validation and training data separately
def plot_loss_curves(history):
  """
  Returns separate loss curves for training and validation metrics.
  """ 
  loss = history.history['loss']
  val_loss = history.history['val_loss']

  accuracy = history.history['accuracy']
  val_accuracy = history.history['val_accuracy']

  epochs = range(len(history.history['loss']))

  # Plot loss
  plt.plot(epochs, loss, label='training_loss')
  plt.plot(epochs, val_loss, label='val_loss')
  plt.title('Loss')
  plt.xlabel('Epochs')
  plt.legend()

  # Plot accuracy
  plt.figure()
  plt.plot(epochs, accuracy, label='training_accuracy')
  plt.plot(epochs, val_accuracy, label='val_accuracy')
  plt.title('Accuracy')
  plt.xlabel('Epochs')
  plt.legend();

In [ ]:
# Check out our model's performance 
plot_loss_curves(history)

Ok, let's do some further experiment... I'll just add a dropout layer to help with the overfitting.

In [ ]:
# Create the model 
model_2 = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),  # Adding dropout layer for regularization and preventing overfitting
    layers.Dense(10, activation='softmax')
])

# Compile the model
model_2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Fit the model
history_2 = model_2.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val), batch_size=32)

And check:

In [ ]:
plot_loss_curves(history_2)

The figure seems better. Now I want to see what happens if I increase the dropout rate. It should increase the amount of regularization. 

In [ ]:
# Create the model architecture
model_3 = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.6),  # Adding dropout layer for regularization and preventing overfitting
    layers.Dense(10, activation='softmax')
])

# Compile the model
model_3.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Fit the model
history_3 = model_3.fit(X_train, y_train, epochs=12, validation_data=(X_val, y_val), batch_size=32)

In [ ]:
# Check out our model's performance 
plot_loss_curves(history_3)

Let's add early stopping callback. I'll add it to the model_2, with 0.5 dropout rate.

In [ ]:
# Create the model
model_4 = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),  # Adding dropout layer for regularization
    layers.Dense(10, activation='softmax')
])

# Compile the model
model_4.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Define early stopping callback
early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Fit the model with early stopping
history_4 = model_4.fit(X_train, y_train, epochs=50, validation_data=(X_val, y_val), batch_size=32, callbacks=[early_stopping])


In [ ]:
# Check out our model's performance 
plot_loss_curves(history_4)

Let's also try simpler architecture. 

In [ ]:
# Create the model
model_5 = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(6, kernel_size=(5, 5), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(16, kernel_size=(5, 5), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(),
    layers.Dense(120, activation='relu'),
    layers.Dense(84, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Compile the model
model_5.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Fit the model
history_5 = model_5.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val), batch_size=32) 

# Print model summary
model_5.summary()


In [ ]:
# Check out our model's performance 
plot_loss_curves(history_5)

Let's try and add dropout to the model_5

In [ ]:

# Create the model
model_6 = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(6, kernel_size=(5, 5), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(16, kernel_size=(5, 5), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(),
    layers.Dense(120, activation='relu'),
    layers.Dropout(0.5),  # Add dropout with rate 0.5
    layers.Dense(84, activation='relu'), 
    layers.Dropout(0.5),  # Add dropout with rate 0.5
    layers.Dense(10, activation='softmax')
])

# Compile the model
model_6.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Fit the model
history_6 = model_6.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val), batch_size=32)

# Print model summary
model_6.summary()

What could I try next? Maybe try adding early stopping.

In [ ]:
# Create the model
model_7 = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(6, kernel_size=(5, 5), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(16, kernel_size=(5, 5), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(),
    layers.Dense(120, activation='relu'),
    layers.Dropout(0.5),  # Add dropout with rate 0.5
    layers.Dense(84, activation='relu'),
    layers.Dropout(0.5),  # Add dropout with rate 0.5
    layers.Dense(10, activation='softmax')
])

# Compile the model
model_7.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Define early stopping callback
early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model with early stopping
history_7 = model_7.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), batch_size=32, callbacks=[early_stopping])

# Print model summary
model_7.summary()

In [ ]:
plot_loss_curves(history_7)

Let's try something else.

The model might learn to recognize the digits. Data augmentation and adding more training data would be another option, but since preprocessing of the data is not allowed anyway, I will not use my time on doing that. 

So, I'll just fiddle with the model and see, what will happen. 

In [ ]:
# Create the model 
model_9 = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(6, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(16, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(),
    layers.Dense(84, activation='relu'),
    layers.Dropout(0.5),  # Add dropout with rate 0.5
    layers.Dense(120, activation='relu'),
    layers.Dropout(0.5),  # Add dropout with rate 0.5
    layers.Dense(10, activation='softmax')
])

# Compile the model
model_9.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Fit the model
history_9 = model_9.fit(X_train, y_train, epochs=15, validation_data=(X_val, y_val), batch_size=32)

# Print model summary
model_9.summary()


In [ ]:
plot_loss_curves(history_9)

I guess I have tried different scenarios enough. I'm not happy about the model's valuation accuracy. The variation between possible outcomes is too high. I can only decrease the number of parameters as much as I can to get the best possible price with the poor accuracy. So, I'll just use this one. Let's save it.

In [ ]:
model_9.save('MiraVorneExam.keras')